# Bonus Lab: Dask DataFrame

**Module:** Week 3 Day 1 (optional)  
**Kernel:** Shared repo-root environment


> This is an optional bonus walkthrough. You are not expected to master this tool today.
> The goal is awareness: understand what problem this tool tries to solve, when it may help, and what trade-offs it introduces.


Dask keeps a pandas-like API but splits the data into partitions and stays lazy: every operation adds a node to a task graph, and nothing runs until you call `.compute()`. Watch the object types as you go. You are not holding data, you are holding a plan.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

REAL_PATH = Path("data/yellow_tripdata_2024-04.parquet")
SYNTH_DIR = Path("data/synthetic_tripdata_parquet")

if REAL_PATH.exists():
    SOURCE_PATH = REAL_PATH
    print("using real TLC data:", SOURCE_PATH)
else:
    if not any(SYNTH_DIR.glob("*.parquet")):
        rng = np.random.default_rng(42)
        n = 1_000_000
        pickup = pd.to_datetime("2024-04-01") + pd.to_timedelta(
            rng.integers(0, 30 * 24 * 3600, n), unit="s"
        )
        duration_s = rng.integers(60, 5400, n)
        synth = pd.DataFrame(
            {
                "tpep_pickup_datetime": pickup,
                "tpep_dropoff_datetime": pickup + pd.to_timedelta(duration_s, unit="s"),
                "payment_type": rng.integers(0, 5, n),
                "fare_amount": (duration_s / 60 * rng.uniform(2.0, 4.0, n)).round(2),
                "trip_distance": (duration_s / 60 * rng.uniform(0.2, 0.6, n)).round(2),
            }
        )
        SYNTH_DIR.mkdir(parents=True, exist_ok=True)

        parts = np.array_split(synth, 8)
        for i, part in enumerate(parts):
            part.to_parquet(SYNTH_DIR / f"part-{i:02d}.parquet", index=False)

    SOURCE_PATH = SYNTH_DIR
    print("real file not found, using synthetic partitioned data:", SOURCE_PATH)


In [ ]:
import importlib.util

if importlib.util.find_spec("dask") is None:
    raise ImportError(
        "Dask is not installed in this environment. "
        "Install with: uv add 'dask[dataframe]' or skip this optional bonus notebook."
    )

import dask.dataframe as dd

ddf = dd.read_parquet(SOURCE_PATH)
print(type(ddf))
print("npartitions:", ddf.npartitions)
ddf


In [ ]:
ddf = ddf.assign(
    trip_duration_min=(ddf["tpep_dropoff_datetime"] - ddf["tpep_pickup_datetime"]).dt.total_seconds() / 60
)
dask_answer = ddf[ddf["trip_duration_min"] > 25].groupby("payment_type")["fare_amount"].mean()

print(type(dask_answer))
dask_answer

Nothing has been computed yet. `.compute()` executes the whole graph and returns a plain pandas object. One month of taxi data fits in memory, so expect Dask to be near pandas speed or slower here: partitioning has overhead, and its payoff comes when data does not fit in memory or is spread over many files.

In [ ]:
%%time
dask_result = dask_answer.compute()
dask_result

## Your Turn
Change the business question: **for trips longer than 10 minutes, what is the average fare by payment type?**
Build the lazy result, print its type, then `.compute()`.

Hint: copy the previous lazy expression, change `> 25` to `> 10`, then call `.compute()`.


In [ ]:
# Your code here

## Bonus Tool Takeaways

| Topic | What students should remember | Do they need it today? |
|---|---|---|
| Dask | Pandas-like API, lazy execution, partitions, `.compute()` | No, awareness only |
| Modin | Tries to keep Pandas code while changing the execution engine | No, awareness only |
| FireDucks | Pandas-like acceleration, but environment-sensitive | No, awareness only |
| Parquet | Columnar storage, smaller files, column pruning, compression trade-offs | Yes, important for data engineering |
